# 02 — Definitieve voorbereiding zonder grafieken

ZIPs → vaste cleaning → uuraggregatie → features → target → chronologische splits → modelinvoer. Geen trainingsrapporten of modellen nodig.

## Bijdragen — Messi10

**Raoul Buffels:** dataverwerking en cleaning, EDA en hypothesen, features, modellen,
tuning, evaluatie en documentatie; Citi Bike-frontend en backend.
**Jorre Van Dyck:** hulp bij deployment op Render.
Codex hielp met implementatie, uitleg, tests en uitvoering. Ieder teamlid moet de
eigen keuzes mondeling kunnen uitleggen.

We laden alleen bronverwerking, features en voorbereiding. Er wordt geen plot- of modelmodule geïmporteerd.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display
locations=[Path.cwd(),Path.cwd().parent,Path.cwd()/'Citybike_full']
ROOT=next(p for p in locations if (p/'citibike').exists())
sys.path.insert(0,str(ROOT))
from citibike.config import SOURCE
from citibike.model_data import prepare_model_data,load_model_data,CONTRACT
from citibike.features import FEATURES,leakage_audit

## Cleaning → aggregatie → features → target

De volgende stap controleert bron- en cachechecksums, harmoniseert schema’s, verwijdert bewezen bronkopieën en bewaart kwaliteitsflags. Geldige starts worden per lokaal uur geteld. Ontbrekende maanden veroorzaken een fout.

Target: geregistreerde vertrekken per uurvak. Kalenderfeatures zijn vooraf bekend; daglags gebruiken afgesloten dagen. Rolling wordt eerst verschoven. De eerste veertien dagen missen historie en vallen uit modeldata. DST gebruikt 24 lokale labels met 0 of 2 fysieke uren waar nodig. Afwijkende bestaande modeldata worden niet overschreven.

In [2]:
prepared=prepare_model_data(SOURCE)
display(pd.DataFrame([CONTRACT]))

Inventariscache 1/175: 2013-citibike-tripdata.zip


Inventariscache 2/175: 2014-citibike-tripdata.zip


Inventariscache 3/175: 2015-citibike-tripdata.zip


Inventariscache 4/175: 2016-citibike-tripdata.zip


Inventariscache 5/175: 2017-citibike-tripdata.zip


...

Cache NYC 2026-06


Cache NYC 2026-07


Cache NYC 2026-08


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,target,area,prediction_time,history_days,features,window_years,train_end_exclusive,validation_end_exclusive,test_start,test_end_exclusive
0,hourly_demand,NYC,00:00 America/New_York op voorspeldag D,14,"[hour_sin, hour_cos, weekday_sin, weekday_cos,...",None,2026-05-01,2026-07-01,2026-07-01,2026-09-01


## Splits en voorbereide uitvoer

Training: vóór mei; validatie: mei–juni. De testpartitions juli–augustus blijven afgeschermd. Deze voorbereiding vereist geen evaluatie-lock en leest geen testlabels. De vijf folds komen alleen uit training.

In [3]:
training,validation,prepared=load_model_data()
display(pd.DataFrame([{k:v for k,v in prepared.items() if k not in ['folds','contract']}]))
display(pd.DataFrame(prepared['folds']))
display(training[['timestamp','rides']+FEATURES].tail(3))
display(validation[['timestamp','rides']+FEATURES].head(3))

,prepared_at,path,sha256,dataset_fingerprint,existing_dataset_equal,rows,train_rows,validation_rows,train_start,train_end_exclusive,validation_start,validation_end_exclusive,test_start,test_end_exclusive,fold_fingerprint,training_results_required,test_labels_read,test_preparation
0,2026-10-14T15:08:33.145076+00:00,C:\Deeplearning\CloudAI\Citybike_full\data\mod...,2be4c1fa94f1ee4657020c8912af03e6aadd7735d8ba0d...,409eb1829fa0565f464bbd031cc4e17c8c64f7c9aae466...,True,114336,112872,1464,2013-06-15 00:00:00,2026-05-01,2026-05-01,2026-07-01,2026-07-01,2026-09-01,59238f467d526664e56943f96d102928ede96e4e371fec...,False,False,Ruwe testpartitions technisch opgeschoond; tes...


,train_start,train_stop,validation_start,validation_stop,train_last_day,validation_first_day,validation_last_day
0,0,109512,109512,110184,2025-12-11,2025-12-12,2026-01-08
1,0,110184,110184,110856,2026-01-08,2026-01-09,2026-02-05
2,0,110856,110856,111528,2026-02-05,2026-02-06,2026-03-05
3,0,111528,111528,112200,2026-03-05,2026-03-06,2026-04-02
4,0,112200,112200,112872,2026-04-02,2026-04-03,2026-04-30


,timestamp,rides,hour_sin,hour_cos,weekday_sin,weekday_cos,year_sin,year_cos,is_weekend,clock_hours,lag_1d,lag_2d,lag_7d,lag_14d,same_hour_mean_7d,previous_day_total
112869,2026-04-30 21:00:00,5418,-0.707107,0.707107,0.433884,-0.900969,0.888682,-0.458524,0,1,2645.0,5156.0,6557.0,7253.0,3941.714286,141454.0
112870,2026-04-30 22:00:00,4439,-0.500000,0.866025,0.433884,-0.900969,0.888682,-0.458524,0,1,1449.0,3939.0,5855.0,5865.0,3102.857143,141454.0
112871,2026-04-30 23:00:00,2875,-0.258819,0.965926,0.433884,-0.900969,0.888682,-0.458524,0,1,620.0,2571.0,3667.0,3831.0,2091.142857,141454.0


,timestamp,rides,hour_sin,hour_cos,weekday_sin,weekday_cos,year_sin,year_cos,is_weekend,clock_hours,lag_1d,lag_2d,lag_7d,lag_14d,same_hour_mean_7d,previous_day_total
0,2026-05-01 00:00:00,1702,0.000000,1.000000,-0.433884,-0.900969,0.880663,-0.473744,0,1,493.0,1196.0,2123.0,2295.0,1271.000000,139724.0
1,2026-05-01 01:00:00,948,0.258819,0.965926,-0.433884,-0.900969,0.880663,-0.473744,0,1,166.0,545.0,1068.0,1249.0,740.714286,139724.0
2,2026-05-01 02:00:00,523,0.500000,0.866025,-0.433884,-0.900969,0.880663,-0.473744,0,1,136.0,383.0,682.0,708.0,492.714286,139724.0


## Beschikbaarheid om 00:00

Geen telling op de voorspeldag beïnvloedt eerdere features. Historische tellingen moeten tijdig beschikbaar zijn; maandarchieven bewijzen geen live feed. De modelnotebooks laden exact deze output.

In [4]:
display(pd.DataFrame(leakage_audit()))

,feature,available_at,available_at_prediction,availability_assumption,target_information,leaks
0,hour_sin,kalender bekend vóór 00:00 op D,True,kalender deterministisch bekend,geen doelwaarden,False
1,hour_cos,kalender bekend vóór 00:00 op D,True,kalender deterministisch bekend,geen doelwaarden,False
2,weekday_sin,kalender bekend vóór 00:00 op D,True,kalender deterministisch bekend,geen doelwaarden,False
3,weekday_cos,kalender bekend vóór 00:00 op D,True,kalender deterministisch bekend,geen doelwaarden,False
4,year_sin,kalender bekend vóór 00:00 op D,True,kalender deterministisch bekend,geen doelwaarden,False
5,year_cos,kalender bekend vóór 00:00 op D,True,kalender deterministisch bekend,geen doelwaarden,False
6,is_weekend,kalender bekend vóór 00:00 op D,True,kalender deterministisch bekend,geen doelwaarden,False
7,clock_hours,kalender bekend vóór 00:00 op D,True,kalender deterministisch bekend,geen doelwaarden,False
8,lag_1d,uitsluitend volledige dagen vóór D; rolling ee...,voorwaardelijk: externe tellingen bekend vóór D,Offline backtest veronderstelt tijdige externe...,historische doelwaarden toegestaan,False
9,lag_2d,uitsluitend volledige dagen vóór D; rolling ee...,voorwaardelijk: externe tellingen bekend vóór D,Offline backtest veronderstelt tijdige externe...,historische doelwaarden toegestaan,False
